In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import StringIO

In [3]:
DATA_DIR = Path.cwd().resolve().parent / "datos"

df_bank = pd.read_parquet(DATA_DIR / "02_datos_con_tipo_de_dato_ajustado_banko.parquet")

In [4]:
cols_categoricas = ['job', 'marital', 'default', 'housing',
       'loan', 'contact', 'month', 'poutcome']

df_bank[cols_categoricas] = df_bank[cols_categoricas].astype('category')

df_bank['education'] = pd.Categorical(
    df_bank['education'], 
    categories=['primary', 'secondary', 'tertiary','unknown'], 
    ordered=True)

cols_categoricas_ordinales = ['education']

In [5]:
cols_numericas_decimales = ['balance']

df_bank[cols_numericas_decimales] = df_bank[cols_numericas_decimales].astype('float64')

En la exploración y analisis de los datos, se identificó que la variable duration puede generar ruido en la definición del modelo, ya que no podemos predecir si llamar al cliente o no en función a la duración de la llamada que no hemos realizado. Ya que esta duración solo es conocida después de la llamada. Por tal, no es un criterio para definir las campañas.

In [6]:
#cols_numericas_enteros = ['age', 'day', 'duration','campaign','pdays', 'previous']

cols_numericas_enteros = ['age', 'day','campaign','pdays', 'previous']

df_bank[cols_numericas_enteros] = df_bank[cols_numericas_enteros].astype('int64')

Herramientas para el procesamiento

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [8]:
numeric_pipeline_median = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

numeric_pipeline_mean_scaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())  # Escalamos en este enfoque
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

categorical_ord_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

In [9]:
preprocessor_median = ColumnTransformer(transformers=[
    ('num', numeric_pipeline_median, cols_numericas_enteros),
    ('dec',numeric_pipeline_median,cols_numericas_decimales),
    ('cat', categorical_pipe, cols_categoricas),
    ('cat_ord', categorical_ord_pipe, cols_categoricas_ordinales)
])

preprocessor_median

,transformers,"[('num', ...), ('dec', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [10]:
preprocessor_mean_scale = ColumnTransformer(transformers=[
    ('num', numeric_pipeline_mean_scaled, cols_numericas_enteros),
    ('dec',numeric_pipeline_mean_scaled,cols_numericas_decimales),
    ('cat', categorical_pipe, cols_categoricas),
    ('cat_ord', categorical_ord_pipe, cols_categoricas_ordinales)
])

preprocessor_mean_scale

,transformers,"[('num', ...), ('dec', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


Dividimos 80% para entrenamiento y 20% para test

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_features = df_bank.drop(columns=['y'])
y_target = df_bank['y']

X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    y_target, 
    test_size=0.2, 
    random_state=42
)   

In [13]:
preprocessor_mean_scale.fit(X_test)

,transformers,"[('num', ...), ('dec', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [14]:
feature_names = preprocessor_mean_scale.get_feature_names_out()

x_test_transformed = preprocessor_mean_scale.transform(X_test)

In [15]:
feature_names

array(['num__age', 'num__day', 'num__campaign', 'num__pdays',
       'num__previous', 'dec__balance', 'cat__job_admin.',
       'cat__job_blue-collar', 'cat__job_entrepreneur',
       'cat__job_housemaid', 'cat__job_management', 'cat__job_retired',
       'cat__job_self-employed', 'cat__job_services', 'cat__job_student',
       'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown',
       'cat__marital_divorced', 'cat__marital_married',
       'cat__marital_single', 'cat__default_no', 'cat__default_yes',
       'cat__housing_no', 'cat__housing_yes', 'cat__loan_no',
       'cat__loan_yes', 'cat__contact_cellular', 'cat__contact_telephone',
       'cat__contact_unknown', 'cat__month_apr', 'cat__month_aug',
       'cat__month_dec', 'cat__month_feb', 'cat__month_jan',
       'cat__month_jul', 'cat__month_jun', 'cat__month_mar',
       'cat__month_may', 'cat__month_nov', 'cat__month_oct',
       'cat__month_sep', 'cat__poutcome_failure', 'cat__poutcome_other',
       'cat__poutco

In [16]:
x_test_transformed

array([[-0.09018172,  0.02874778, -0.58403685, ...,  0.        ,
         1.        ,  1.        ],
       [ 0.56893307, -0.81373512, -0.25155682, ...,  0.        ,
         1.        ,  1.        ],
       [-1.50257057,  0.51016658, -0.58403685, ...,  0.        ,
         1.        ,  2.        ],
       ...,
       [ 1.03972936,  1.23229477, -0.58403685, ...,  0.        ,
         1.        ,  1.        ],
       [-0.09018172,  0.02874778, -0.25155682, ...,  0.        ,
         1.        ,  1.        ],
       [-0.37265949,  0.51016658, -0.25155682, ...,  0.        ,
         1.        ,  1.        ]], shape=(9043, 47))

In [17]:
x_test_tranformed = pd.DataFrame(x_test_transformed, columns=feature_names)

x_test_tranformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9043 entries, 0 to 9042
Data columns (total 47 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   num__age                9043 non-null   float64
 1   num__day                9043 non-null   float64
 2   num__campaign           9043 non-null   float64
 3   num__pdays              9043 non-null   float64
 4   num__previous           9043 non-null   float64
 5   dec__balance            9043 non-null   float64
 6   cat__job_admin.         9043 non-null   float64
 7   cat__job_blue-collar    9043 non-null   float64
 8   cat__job_entrepreneur   9043 non-null   float64
 9   cat__job_housemaid      9043 non-null   float64
 10  cat__job_management     9043 non-null   float64
 11  cat__job_retired        9043 non-null   float64
 12  cat__job_self-employed  9043 non-null   float64
 13  cat__job_services       9043 non-null   float64
 14  cat__job_student        9043 non-null   

In [18]:
x_test_tranformed

,num__age,num__day,num__campaign,num__pdays,num__previous,dec__balance,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,...,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep,cat__poutcome_failure,cat__poutcome_other,cat__poutcome_success,cat__poutcome_unknown,cat_ord__education
0,-0.090182,0.028748,-0.584037,-0.411810,-0.288514,-0.251495,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,0.568933,-0.813735,-0.251557,-0.411810,-0.288514,0.727425,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,-1.502571,0.510167,-0.584037,-0.411810,-0.288514,-0.264914,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
3,0.098137,-0.813735,-0.584037,2.951021,0.208445,0.129657,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0
4,1.416366,0.630521,-0.251557,-0.411810,-0.288514,-0.367471,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9038,0.568933,-0.934090,-0.584037,-0.411810,-0.288514,0.167038,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
9039,-0.843456,-0.091607,0.080923,-0.411810,-0.288514,-0.367471,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
9040,1.039729,1.232295,-0.584037,-0.411810,-0.288514,-0.436800,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
9041,-0.090182,0.028748,-0.251557,-0.411810,-0.288514,-0.186639,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


Modelos a utilizar:


 -LGBMClassifier
 
 -RandomForest

In [19]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [20]:
def resumen_clasificacion(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label='yes')
    recall = recall_score(y_test, y_pred, pos_label='yes')
    f1 = f1_score(y_test, y_pred, pos_label='yes')
    
    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
        }

In [21]:
modelos = {
    'LGBM': LGBMClassifier(random_state=42, max_iter=100),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

In [22]:
pipelines = {}

for modelo_nombre, modelo in modelos.items():
    # Pipeline con imputación de la median - sin escalado
    pipelines[f"{modelo_nombre}_median"] = Pipeline(steps=[
        ('preprocessor', preprocessor_median),
        ('classifier', modelo)
    ])

    # Pipeline con imputación de la media y escalado
    pipelines[f"{modelo_nombre}_mean_scaled"] = Pipeline(steps=[
        ('preprocessor', preprocessor_mean_scale),
        ('classifier', modelo)
    ])

In [23]:
pipelines

{'LGBM_median': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median'))]),
                                                   ['age', 'day', 'campaign',
                                                    'pdays', 'previous']),
                                                  ('dec',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median'))]),
                                                   ['balance']),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
  

Entrenamiento y evaluacion de los Pipelines

In [24]:
resultados = {}

for nombre_pipeline, pipeline in pipelines.items():
    # Entrenamiento
    pipeline.fit(X_train, y_train)
    
    # Predicción
    y_pred = pipeline.predict(X_test)
    
    # Guardar resultados
    resultados[nombre_pipeline] = resumen_clasificacion(y_test, y_pred)

[LightGBM] [Warning] num_iterations is set=100, max_iter=100 will be ignored. Current value: num_iterations=100
[LightGBM] [Info] Number of positive: 4198, number of negative: 31970
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 774
[LightGBM] [Info] Number of data points in the train set: 36168, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116069 -> initscore=-2.030190
[LightGBM] [Info] Start training from score -2.030190


c:\Users\aical\Documents\JM_SD_SC_marketing_bancario\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] num_iterations is set=100, max_iter=100 will be ignored. Current value: num_iterations=100
[LightGBM] [Info] Number of positive: 4198, number of negative: 31970
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002192 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 36168, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116069 -> initscore=-2.030190
[LightGBM] [Info] Start training from score -2.030190


c:\Users\aical\Documents\JM_SD_SC_marketing_bancario\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
df_resultados = pd.DataFrame(resultados).T.sort_values(by='F1 Score', ascending=False)

df_resultados

,Accuracy,Precision,Recall,F1 Score
LGBM_median,0.894615,0.669118,0.250229,0.364243
LGBM_mean_scaled,0.894172,0.665842,0.246563,0.359866
Random Forest_median,0.891518,0.633495,0.239230,0.347305
Random Forest_mean_scaled,0.891297,0.631068,0.238313,0.345975


En este caso, el mejor modelo lo tomaremos como el mayor F1 score, ya que es el que mejor identifica los casos positivos. Por ello, seleccionamos el modelo LGBM_median. 

La variable duration tenía una alta correlación con la aceptación del producto ofrecido, pero al retirarla nuestro modelo es mas realista, ya que cuando incluía la variable duration utilizaba valores conocidos solo después de la llamada. Ahora el modelo es más útil para elegir a qué clientes llamar. 

Para robustecer el modelo se recomendaría utilizar variables adicionales con nueva información.

In [28]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

In [29]:
pipeline_lgbm = Pipeline(steps=[
        ('preprocessor', preprocessor_median),
        ('classifier',LGBMClassifier(random_state=42))
    ])

In [48]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 7],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

f1_scorer = make_scorer(f1_score, pos_label='yes')

grid_search = GridSearchCV(
    estimator=pipeline_lgbm,
    param_grid=param_grid,
    cv=10,
    scoring=f1_scorer,
    n_jobs=-1
)

In [49]:
grid_search.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 4198, number of negative: 31970
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006910 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 774
[LightGBM] [Info] Number of data points in the train set: 36168, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116069 -> initscore=-2.030190
[LightGBM] [Info] Start training from score -2.030190
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__colsample_bytree': [0.8, 1.0], 'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [5, 7], 'classifier__n_estimators': [100, 200], ...}"
,scoring,make_scorer(f...pos_label=yes)
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('dec', ...), ...]"


In [50]:
grid_search.best_params_

{'classifier__colsample_bytree': 1.0,
 'classifier__learning_rate': 0.1,
 'classifier__max_depth': 7,
 'classifier__n_estimators': 200,
 'classifier__subsample': 0.8}

In [51]:
grid_search.best_score_

np.float64(0.3554684477734783)

Guardamos el modelo (Este fue nuestro modelo definitivo).

In [53]:
import joblib

best_model = grid_search.best_estimator_

DATA_DIR = Path.cwd().resolve().parent / "modelos"

DATA_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, DATA_DIR / "ligtgbm_best_model.joblib")

['C:\\Users\\aical\\Documents\\JM_SD_SC_marketing_bancario\\modelos\\ligtgbm_best_model.joblib']